# Fase 3 — Análisis exploratorio (nivel compuesto)

| Campo | Valor |
|---|---|
| **Entrada compuesto** | `compounds_features.csv` (107 filas) |
| **Entrada medición** | `activities_clean.csv` (perfil dianas/endpoints) |
| **Doc** | [`docs/analisis_proyecto/fases/fase3_eda.md`](../docs/fases/fase3_eda.md) |

> **Unidad de análisis:** descriptores fisicoquímicos sobre **107 compuestos**, no 2.807 filas. Reportar siempre **n por familia** (Carbamates tiene muy pocos compuestos).


## 0. Configuración

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Image, display

from src.paths import PROJECT_ROOT as ROOT, setup_path
setup_path()

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams.update({"figure.figsize": (10, 5), "figure.dpi": 120})

FIG_DIR = ROOT / "outputs" / "chembl" / "figures"
RESULTS_DIR = ROOT / "outputs" / "chembl" / "results"
for d in (FIG_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

from src.analisis_proyecto.chembl_preprocessing import (
    FEATURE_COLS,
    correlation_with_target,
    load_bioactivity,
    summary_statistics,
)

ACTIVITIES_CSV = ROOT / "data" / "processed" / "activities_clean.csv"
COMPOUNDS_CSV = ROOT / "data" / "processed" / "compounds_features.csv"

assert COMPOUNDS_CSV.exists(), "Ejecuta fase2_limpieza.ipynb primero"

activities = load_bioactivity(ACTIVITIES_CSV)
compounds = pd.read_csv(COMPOUNDS_CSV)
print(f"Compuestos: {compounds.shape} | Mediciones: {activities.shape}")
print("n por familia (compuesto):")
display(compounds["family"].value_counts().to_frame("n_compuestos"))


## 1. Tendencia central (107 compuestos)

In [ ]:
stats_table = summary_statistics(compounds, FEATURE_COLS + ["pchembl_median"])
display(stats_table.round(4))

for col in FEATURE_COLS:
    fig, ax = plt.subplots()
    compounds[col].dropna().hist(bins=20, ax=ax, edgecolor="white", color="#4C72B0")
    ax.set_title(f"{col} — {len(compounds)} compuestos")
    ax.set_xlabel(col)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"hist_{col}.png", bbox_inches="tight")
    plt.show()


## 2. Boxplots por familia (n anotado)

In [ ]:
family_counts = compounds["family"].value_counts()

for col in FEATURE_COLS + ["pchembl_median"]:
    fig, ax = plt.subplots(figsize=(12, 5))
    order = compounds.groupby("family")[col].median().sort_values().index
    sns.boxplot(data=compounds, x="family", y=col, order=order, ax=ax)
    labels = [f"{f}\n(n={family_counts[f]})" for f in order]
    ax.set_xticklabels(labels, rotation=30, ha="right")
    ax.set_title(f"{col} por familia química")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"box_{col}_by_family.png", bbox_inches="tight")
    plt.show()

print("Advertencia: familias con n<5 compuestos tienen comparaciones poco estables.")


## 3. Promiscuidad y perfil de dianas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
compounds["n_targets"].hist(bins=20, ax=axes[0], edgecolor="white")
axes[0].set_title("Distribución n_targets por compuesto")
sns.boxplot(data=compounds, x="family", y="n_targets", ax=axes[1])
axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig(FIG_DIR / "promiscuity_distribution.png", bbox_inches="tight")
plt.show()

if "target_type" in activities.columns:
    ct = pd.crosstab(activities["chembl_id"], activities["target_type"])
    top = ct.sum(axis=0).nlargest(15).index
    fig, ax = plt.subplots(figsize=(14, 10))
    sns.heatmap(ct[top], cmap="YlOrRd", ax=ax)
    ax.set_title("Heatmap compuesto × target_type")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "heatmap_compound_target.png", bbox_inches="tight")
    plt.show()


## 4. Perfil de endpoints

Hay **13 `standard_type`** distintos (Ki, IC50, EC50, …). Por eso **no** se agrupa `pchembl_value` crudo como target único de regresión.


In [ ]:
if "standard_type" in activities.columns:
    display(activities["standard_type"].value_counts().to_frame("n_filas"))

if "standard_relation" in activities.columns:
    print("Relaciones de censura en mediciones:")
    display(activities["standard_relation"].value_counts(dropna=False).to_frame("n"))


## 5. Correlación honesta (pchembl_median a nivel compuesto)

In [ ]:
corr_table = correlation_with_target(compounds, target="pchembl_median", columns=FEATURE_COLS)
display(corr_table.round(4))

corr_matrix = compounds[FEATURE_COLS + ["pchembl_median"]].corr(method="pearson")
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlación Pearson — 107 compuestos")
plt.tight_layout()
plt.savefig(FIG_DIR / "correlation_heatmap.png", bbox_inches="tight")
plt.show()

pair_cols = FEATURE_COLS[:4] + ["pchembl_median"]
g = sns.pairplot(compounds[pair_cols].dropna(), diag_kind="hist", corner=True)
g.fig.suptitle("Scatter matrix — nivel compuesto", y=1.02)
g.savefig(FIG_DIR / "correlation_pairplot.png", bbox_inches="tight")
plt.show()


---
*Anterior:* [`fase2_limpieza.ipynb`](fase2_limpieza.ipynb)  
*Siguiente:* [`fase4_modelado.ipynb`](fase4_modelado.ipynb)
